# Analysis of chemical components

MedTourEasy / DataCamp cosmetics brief. Catalogue stats in pandas + SQLite,
then a one-hot ingredient matrix, t-SNE map, and cosine neighbors for
dry-skin moisturizers.


In [1]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

ROOT = None
for cand in [Path(".."), Path("."), Path("/workspace/artifacts/cosmetics-ingredient-analysis")]:
    if (cand / "data" / "cosmetics.csv").exists():
        ROOT = cand
        break
df = pd.read_csv(ROOT / "data" / "cosmetics.csv")
print(df.shape, "brands", df.Brand.nunique())
display(df["Label"].value_counts().to_frame("n"))


(1472, 11) brands 116


,n
Label,
Moisturizer,298
Cleanser,281
Face Mask,266
Treatment,248
Eye cream,209
Sun protect,170


## Catalogue (SQL)

In [2]:
con = sqlite3.connect(":memory:")
df.to_sql("cosmetics", con, index=False, if_exists="replace")
display(pd.read_sql("SELECT Label, COUNT(*) n, ROUND(AVG(Price),1) mean_price, ROUND(AVG(Rank),2) mean_rank FROM cosmetics GROUP BY Label ORDER BY n DESC", con))
display(pd.read_sql("SELECT Brand, COUNT(*) n FROM cosmetics GROUP BY Brand ORDER BY n DESC LIMIT 8", con))
display(pd.read_sql("SELECT SUM(Dry) dry, SUM(Sensitive) sensitive, SUM(CASE WHEN Combination+Dry+Normal+Oily+Sensitive=0 THEN 1 ELSE 0 END) no_flag FROM cosmetics", con))
con.close()


,Label,n,mean_price,mean_rank
0,Moisturizer,298,69.1,4.24
1,Cleanser,281,32.6,4.31
2,Face Mask,266,42.6,4.17
3,Treatment,248,79.2,4.22
4,Eye cream,209,63.6,3.81
5,Sun protect,170,45.9,4.05


,Brand,n
0,CLINIQUE,79
1,SEPHORA COLLECTION,66
2,SHISEIDO,63
3,ORIGINS,54
4,MURAD,47
5,PETER THOMAS ROTH,46
6,KIEHL'S SINCE 1851,46
7,FRESH,44


,dry,sensitive,no_flag
0,904,756,471


## Filter moisturizers for dry skin

In [3]:
md = df[(df.Label=="Moisturizer") & (df.Dry==1)].reset_index(drop=True)
print(len(md), "dry moisturizers")


190 dry moisturizers


## Tokenize + document-term matrix

In [4]:
corpus, ingredient_idx, idx = [], {}, 0
for i in range(len(md)):
    tokens = md.loc[i, "Ingredients"].lower().split(", ")
    corpus.append(tokens)
    for t in tokens:
        if t not in ingredient_idx:
            ingredient_idx[t] = idx
            idx += 1
print("vocab", len(ingredient_idx), "decyl oleate", ingredient_idx["decyl oleate"])
A = np.zeros((len(md), len(ingredient_idx)))
for i, tokens in enumerate(corpus):
    for t in tokens:
        A[i, ingredient_idx[t]] = 1
print("A", A.shape, "mean ingredients", round(A.sum(1).mean(), 1))


vocab 2233 decyl oleate 25
A (190, 2233) mean ingredients 35.1


## t-SNE + cosine neighbors

In [5]:
xy = TSNE(n_components=2, learning_rate=200, random_state=42, perplexity=30).fit_transform(A)
md = md.copy(); md["X"] = xy[:,0]; md["Y"] = xy[:,1]
sim = cosine_similarity(A)
p1 = "Color Control Cushion Compact Broad Spectrum SPF 50+"
p2 = "BB Cushion Hydra Radiance SPF 50"
i1 = md.index[md.Name==p1][0]; i2 = md.index[md.Name==p2][0]
print("cosine", round(float(sim[i1,i2]), 3))
s1, s2 = set(corpus[i1]), set(corpus[i2])
print("shared", len(s1 & s2), "jaccard", round(len(s1&s2)/len(s1|s2), 3))
rows = []
for j in np.argsort(-sim[i1])[:6]:
    rows.append({"brand": md.loc[j,"Brand"], "name": md.loc[j,"Name"],
                 "price": int(md.loc[j,"Price"]), "rank": float(md.loc[j,"Rank"]),
                 "cosine": round(float(sim[i1,j]), 3)})
display(pd.DataFrame(rows))


cosine 0.535
shared 23 jaccard 0.365


,brand,name,price,rank,cosine
0,AMOREPACIFIC,Color Control Cushion Compact Broad Spectrum S...,60,4.0,1.000
1,LANEIGE,BB Cushion Hydra Radiance SPF 50,38,4.3,0.535
2,SUPERGOOP!,CC Cream Daily Correct Broad Spectrum SPF 35+ ...,34,4.4,0.333
3,IT COSMETICS,Your Skin But Better™ CC+Illumination™ Cream w...,38,3.9,0.288
4,LANEIGE,Water Sleeping Mask,25,4.4,0.279
5,IT COSMETICS,Your Skin But Better™ CC+™ Cream with SPF 50+,38,4.1,0.259


## Takeaways

- Laneige BB cushion is the nearest formula neighbor of the AmorePacific cushion (cosine 0.535) and is $22 cheaper.
- t-SNE is for looking. Cosine on the binary matrix is the actual score.
- 471 SKUs have no skin-type flag. Do not treat those 0/1 columns as complete.
